In [1]:
from pathlib import Path

import logfire

from src.common.cache.embedding_cache import EmbeddingCache
from src.common.services.qdrant import QdrantStorageService
from src.common.storage.storage_factory import StorageFactory
from src.common.utils.config import config
from src.common.utils.constants import ParseMethod, StorageType
from src.common.utils.helper import separate_content
from src.common.utils.tokenizer import TikTokenTokenizer
from src.ingestion.embedding import EmbeddingService
from src.ingestion.processor import Processor

/home/mano/Manoj/Learning/k_academy/advanced_rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
logfire.configure(service_name="parsing")

Logfire project URL: https://logfire-us.pydantic.dev/manojee/studious

In [3]:
cwd = Path.cwd().parent
file_path = cwd / "data/Attention-is_all_you_need.pdf"

In [4]:
local_config = {
    "type": StorageType.LOCAL.value,
    "base_dir": config.STORAGE_BASE_DIR,
}

in_storage = StorageFactory.create(local_config)

In [5]:
tokenizer = TikTokenTokenizer()
emb_cache = await EmbeddingCache.create(dsn=config.POSTGRES_CONN_STRING, max_entries=50_000)
embedding_service = EmbeddingService(
    model_name=config.EMBEDDING_MODEL_NAME,
    dimensions=config.EMBEDDING_DIMENSIONS,
    batch_size=config.EMBEDDING_BATCH_SIZE,
    cache=emb_cache,
)

storage_service = QdrantStorageService(
    url=config.QDRANT_CLUSTER_ENDPOINT,
    vector_size=embedding_service.vector_size,
    collection_name=config.QDRANT_COLLECTION_NAME,
)

In [6]:
processor = Processor(tokenizer, embedding_service, storage_service)

In [ ]:
out, doc_id = await processor.process_document(
    file_path=str(file_path), parse_method=ParseMethod.DOCLING
)

In [ ]:
print(out)

In [ ]:
content_list, multimodal_items, text_blocks = separate_content(out)

In [ ]:
multimodal_items

In [ ]:
chunk_context = await processor._chunk_doc_content(
    file_path=file_path,
    content_list=content_list,
    multimodal_items=multimodal_items,
    doc_id=doc_id,
    parse_method=ParseMethod.DOCLING,
    text_blocks=text_blocks,
)

In [8]:
result = await processor.ingest_document(file_path=file_path, parse_method=ParseMethod.DOCLING)

12:45:46.852 Starting document parsing: docling - /home/mano/Manoj/Learning/k_academy/advanced_rag/data/Attention-is_all_you_need.pdf
12:45:46.862 Cache HIT - key=48563605… file=/home/mano/Manoj/Learning/k_academy/advanced_rag/data/Attention-is_all_you_need.pdf
12:45:46.862 Cache HIT - Returning cached result for /home/mano/Manoj/Learning/k_academy/advanced_rag/data/Attention-is_all_you_need.pdf
12:45:46.863 Stage 1 complete: 29 content list
12:45:46.863 Content separation complete:
12:45:46.864   - Text content length: 36931 characters
12:45:46.864   - Multimodal items count: 10
12:45:46.864   - Multimodal type distribution: {'image': 6, 'table': 4}
12:45:46.865 Starting chunking with strategy: recursive_character - docling
12:45:46.931 Multimodal chunking complete: 11 chunks from 10 items
12:45:46.932 Chunking complete: 38 chunks produced from 36931 blocks
12:45:46.945 Build parent child chunk: 38
12:45:46.946 Document chunking completed: 38
12:45:46.969 Cache hit ratio: 38/38 (100.0

Currently retrying 1 failed export(s) (2208 bytes)
Failed to export span batch code: None, reason: HTTPSConnectionPool(host='logfire-us.pydantic.dev', port=443): Read timed out. (read timeout=9.99999475479126)


In [ ]:
result